#### 04: Robustness Score

**Measuring each model's stability under input perturbations by computing perplexity shift across typo, word deletion, and synonym substitution attacks.**

In [ ]:
import sys
import os
import yaml
import pandas as pd
from dataclasses import asdict
import json
sys.path.append(os.path.abspath(".."))
from src import robustness_scoring
# import robustness_scoring
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

In [2]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
models=config['models']['list']

In [3]:
#Loading 100 sentences from SST-2
sst2=load_dataset("sst2", split="validation")
sentences=[row["sentence"] for row in sst2.select(range(100))]

In [4]:
results = [robustness_scoring.evaluate_robustness(m, sentences) for m in models]
print(f"\nEvaluated {len(results)} models.")


Evaluating: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

scoring sentences: 100%|██████████| 100/100 [12:06<00:00,  7.26s/it]


Robustness: 0.4997 | mean shift: 0.5003
per type: {'typo': 0.4199, 'deletion': 0.5278, 'synonym': 0.5513}

Evaluating: microsoft/phi-1_5


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/3.29G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

scoring sentences: 100%|██████████| 100/100 [15:23<00:00,  9.24s/it]


Robustness: 0.357 | mean shift: 0.643
per type: {'typo': 0.4107, 'deletion': 0.3089, 'synonym': 0.3514}

Evaluated 5 models.


In [6]:
os.makedirs("results",exist_ok=True)
with open("results/transparency_scores.json","w") as f:
    json.dump({"robustness": results}, f, indent=2)

**Conclusion**

* `TinyLlama-1.1B-Chat` is the most robust at **0.4997**, performing consistently across all perturbation types with no critical failure point.
* `Qwen2-0.5B` is the least robust at **0.076**, with a synonym score of **0.0** and mean shift of **0.9239**, indicating complete embedding collapse under semantic rephrasing.
* Synonym substitution is the most discriminative perturbation type, ranging from **0.0 to 0.55** across models.
* Deletion scores are consistently higher across all models **(0.15 to 0.53)**, suggesting word removal is better tolerated regardless of architecture.
* Model size does not correlate with robustness. `TinyLlama-1.1B` outperforms larger models `Phi-1.5` and `Qwen2-0.5B`, pointing to training quality as the stronger determinant.

**Next: `05_explainability_score.ipynb`** Measures how interpretable each model's predictions are by computing SHAP token attribution scores over a set of input sentences.